## Lesson 3b: Multi-Class Classification (Subcellular Localization)
### What you'll learn
- Adapt the Lesson 3 fine-tuning recipe from BINARY to MULTI-CLASS.
- It takes only three changes:
  1. swap the dataset to `proteinea/localization`,
  2. set `NUM_LABELS` = number of classes (inferred from the data here),
  3. use `average="weighted"` (not `"binary"`) in the metrics.
- Everything else — tokenizer, `Trainer`, training loop — is identical to Lesson 3.

This is a thin variant of `plm_l3_finetune_classification`; see that notebook for the
detailed walkthrough of the fine-tuning mechanics.

> **Run order matters.** The cells below build on each other. Run them **top to bottom**
> (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit
> `NameError: name 'torch' is not defined`, you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports the libraries and defines the constants the rest of the
notebook relies on. `NUM_LABELS` is left as `None` here and inferred from the dataset in
`main()` — that's the "number-of-classes" step.

In [ ]:
import os
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
DATASET_NAME = "proteinea/localization"
NUM_LABELS = None          # inferred from the dataset in main()
OUTPUT_DIR = "./results/lesson3b"
N_TRAIN = 1000
N_TEST = 200

In [ ]:
# MLflow tracking -> shared local SQLite store.
# (repo is pip-installed via `pip install -e .`, so this import needs no sys.path shim)
import mlflow_utils as mu
mlflow = mu.setup_mlflow()

### `compute_metrics` (function)

Multi-class metrics: accuracy plus precision/recall/F1 averaged with `average="weighted"`,
which weights each class by its support — the right choice for imbalanced multi-class
problems like localization. (Lesson 3, being binary, used `average="binary"`.)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

### `prepare_dataset` (function)

Normalises column names and maps labels to a contiguous `0..K-1` index space. Localization
labels may be class-name strings or integers, so we build a `label<->id` map (from all
splits, so a class seen only in test still gets an id) and remember the names for readable
inference.

In [ ]:
def prepare_dataset():
    raw = load_dataset(DATASET_NAME)

    # proteinea datasets use 'sequences'/'labels'; be tolerant of variants.
    def _norm(split):
        cols = split.column_names
        seq_col = next(c for c in ("sequence", "sequences", "seq") if c in cols)
        lab_col = next(c for c in ("label", "labels", "localization", "class") if c in cols)
        return split.rename_columns({seq_col: "sequence", lab_col: "label"})

    raw = DatasetDict({k: _norm(v) for k, v in raw.items()})

    # Build a stable label<->id map from ALL splits (labels may be str or int).
    all_labels = set()
    for k in raw:
        all_labels |= set(raw[k]["label"])
    uniq = sorted(all_labels, key=lambda x: str(x))
    label2id = {lab: i for i, lab in enumerate(uniq)}
    id2label = {i: lab for lab, i in label2id.items()}

    raw = raw.map(lambda b: {"label": [label2id[x] for x in b["label"]]}, batched=True)
    return raw, label2id, id2label

### `main` (function)

In [ ]:
def main():
    global NUM_LABELS
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print(f"Loading dataset: {DATASET_NAME}")
    raw, label2id, id2label = prepare_dataset()
    NUM_LABELS = len(label2id)                       # <-- number-of-classes
    print(f"Detected {NUM_LABELS} classes: {list(label2id)}")

    raw = raw.shuffle(seed=42)
    test_key = "test" if "test" in raw else (
        "validation" if "validation" in raw else list(raw.keys())[-1]
    )
    train_key = "train" if "train" in raw else list(raw.keys())[0]
    ds = DatasetDict({
        "train": raw[train_key].select(range(min(N_TRAIN, len(raw[train_key])))),
        "test": raw[test_key].select(range(min(N_TEST, len(raw[test_key])))),
    })

    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
    )

    def tokenize(batch):
        return tokenizer(batch["sequence"], truncation=True, max_length=512)

    ds = ds.map(tokenize, batched=True, remove_columns=["sequence"])

    LR, BATCH, WD = 2e-5, 8, 0.01
    run_name = "l3b_esm2_8M_localization"
    params = {
        "model": MODEL_NAME, "dataset": DATASET_NAME,
        "n_train": len(ds["train"]), "n_test": len(ds["test"]),
        "num_labels": NUM_LABELS, "epochs": EPOCHS, "lr": LR,
        "batch_size": BATCH, "weight_decay": WD,
    }

    with mu.run("plm-localization", run_name, params=params,
                tags={"lesson": "plm_l3b", "method": "full_finetune", "task": "multiclass"}):

        args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            fp16=torch.cuda.is_available(),
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=LR,
            per_device_train_batch_size=BATCH,
            per_device_eval_batch_size=BATCH,
            num_train_epochs=EPOCHS,
            weight_decay=WD,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            save_total_limit=2,
            logging_steps=50,
            report_to="mlflow",
            run_name=run_name,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=ds["train"],
            eval_dataset=ds["test"],
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics,
        )

        print("\nFine-tuning...")
        trainer.train()

        print("\nFinal evaluation:")
        metrics = trainer.evaluate()
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

        mlflow.log_metrics({
            f"test_{k.replace('eval_', '')}": float(v)
            for k, v in metrics.items() if isinstance(v, (int, float))
        })

        final_dir = os.path.join(OUTPUT_DIR, "final")
        trainer.save_model(final_dir)
        print(f"\nSaved fine-tuned model to: {final_dir}")

        # Inference: show the top-5 predicted classes for one sequence.
        example = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
        inputs = tokenizer(
            example, return_tensors="pt", truncation=True, max_length=512
        ).to(model.device)
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
        top = torch.topk(probs, k=min(5, NUM_LABELS))
        print("\nExample inference (top classes):")
        for p, idx in zip(top.values.tolist(), top.indices.tolist()):
            print(f"  {str(id2label[idx]):<24} {p:.3f}")

    print(
        """
Things to experiment with:
- Larger pLM: MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
- This notebook = Lesson 3 with three changes: dataset -> proteinea/localization,
  NUM_LABELS inferred from the data, and average="weighted" in compute_metrics.
- View runs in the MLflow UI (see Lesson 3 for the exact command).
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [ ]:
main()

## Run comparison — this lesson's MLflow history

How this run compares to every prior run of the same lesson (bigger model / more epochs accumulate here as you re-run). Generated from the shared local MLflow store.

In [ ]:
# --- Run comparison -----------------------------------------------------------
# Every run of this lesson logs to the shared MLflow store. This chart shows
# how the latest run compares to all prior ones (e.g. more epochs / a bigger
# model). It accumulates as you re-run; a no-op on a fresh checkout.
import mlflow_utils as mu
mu.plot_run_comparison('plm-localization')
